1. Imports
2. Load train.csv
3. Basic inspection
4. Separate X / y
5. Train-validation split
6. Missing-value strategy
7. Feature Engineering
8. Preprocessing
9. Baseline models
10. XGBoost
11. CatBoost
12. Compare models
13. Overfitting check
14. Select best model

❌ Hyperparameter tuning — will do in modular coding 
❌ Original Kaggle test.csv — final prediction stage 

In [28]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
train_df = pd.read_csv("../artifacts/train.csv")

In [3]:
target_column = "Purchase"

X = train_df.drop(columns=[target_column]).copy()
y = train_df[target_column].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (550068, 11)
y shape: (550068,)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (440054, 11)
X_test : (110014, 11)
y_train: (440054,)
y_test : (110014,)


In [5]:
print("Missing values in X_train:")
print(X_train.isnull().sum())

print("\nMissing values in X_test:")
print(X_test.isnull().sum())

Missing values in X_train:
User_ID                            0
Product_ID                         0
Gender                             0
Age                                0
Occupation                         0
City_Category                      0
Stay_In_Current_City_Years         0
Marital_Status                     0
Product_Category_1                 0
Product_Category_2            139044
Product_Category_3            306828
dtype: int64

Missing values in X_test:
User_ID                           0
Product_ID                        0
Gender                            0
Age                               0
Occupation                        0
City_Category                     0
Stay_In_Current_City_Years        0
Marital_Status                    0
Product_Category_1                0
Product_Category_2            34594
Product_Category_3            76419
dtype: int64


In [6]:
X_train["Product_Category_2_Missing"] = ( X_train["Product_Category_2"].isnull().astype(int))

X_test["Product_Category_2_Missing"] = (X_test["Product_Category_2"].isnull().astype(int))

X_train["Product_Category_3_Missing"] = (X_train["Product_Category_3"].isnull().astype(int))

X_test["Product_Category_3_Missing"] = (X_test["Product_Category_3"].isnull().astype(int))

In [8]:
product_frequency = X_train["Product_ID"].value_counts()

X_train["Product_Frequency"] = ( X_train["Product_ID"].map(product_frequency))

X_test["Product_Frequency"] = ( X_test["Product_ID"].map(product_frequency).fillna(0))

In [9]:
user_frequency = X_train["User_ID"].value_counts()

X_train["User_Frequency"] = (X_train["User_ID"].map(user_frequency))

X_test["User_Frequency"] = (X_test["User_ID"].map(user_frequency).fillna(0))

In [12]:
X_train["Age_Gender"] = (X_train["Age"].astype(str)+ "_"+ X_train["Gender"].astype(str))
X_test["Age_Gender"] = (X_test["Age"].astype(str)+ "_"+ X_test["Gender"].astype(str))

X_train["Age_City"] = (X_train["Age"].astype(str)+ "_"+ X_train["City_Category"].astype(str))
X_test["Age_City"] = (X_test["Age"].astype(str)+ "_"+ X_test["City_Category"].astype(str))

X_train["Occupation_City"] = (X_train["Occupation"].astype(str)+ "_"+ X_train["City_Category"].astype(str))
X_test["Occupation_City"] = (X_test["Occupation"].astype(str)+ "_"+ X_test["City_Category"].astype(str))



X_train["Occupation_Age"] = ( X_train["Occupation"].astype(str) + "_" + X_train["Age"].astype(str))
X_test["Occupation_Age"] = ( X_test["Occupation"].astype(str) + "_" + X_test["Age"].astype(str))



In [13]:
X_train = X_train.drop(columns=["User_ID", "Product_ID"])
X_test = X_test.drop(columns=["User_ID", "Product_ID"])

In [15]:
categorical_columns = [
    "Gender",
    "Age",
    "Occupation",
    "City_Category",
    "Stay_In_Current_City_Years",
    "Marital_Status",
    "Product_Category_1",
    "Product_Category_2",
    "Product_Category_3",
    "Age_Gender",
    "Age_City",
    "Occupation_City",
    "Occupation_Age"
]

In [16]:
numerical_columns = [
    "Product_Frequency",
    "User_Frequency",
    "Product_Category_2_Missing",
    "Product_Category_3_Missing"
]

In [19]:
num_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

cat_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [20]:
preprocessor = ColumnTransformer(transformers=[("num", num_pipeline, numerical_columns),("cat", cat_pipeline, categorical_columns) ])

In [22]:
X_train_transformed = preprocessor.fit_transform(X_train)

X_test_transformed = preprocessor.transform(X_test)

print("X_train transformed:", X_train_transformed.shape)
print("X_test transformed :", X_test_transformed.shape)

X_train transformed: (440054, 328)
X_test transformed : (110014, 328)


In [25]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Decision Tree": DecisionTreeRegressor(max_depth=20,random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=30, max_depth=15, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=50,max_depth=5,random_state=42)}

In [26]:
results = {}

for name, model in models.items():

    print(f"\nTraining {name}...")

    model.fit(X_train_transformed, y_train)

    y_pred = model.predict(X_test_transformed)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    results[name] = {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2 Score": r2
    }

    print(f"MAE      : {mae:.2f}")
    print(f"RMSE     : {rmse:.2f}")
    print(f"R2 Score : {r2:.4f}")


Training Linear Regression...
MAE      : 2159.85
RMSE     : 2877.76
R2 Score : 0.6704

Training Ridge...
MAE      : 2159.84
RMSE     : 2877.74
R2 Score : 0.6704

Training Decision Tree...
MAE      : 2120.78
RMSE     : 2960.37
R2 Score : 0.6512

Training Random Forest...
MAE      : 2053.25
RMSE     : 2735.38
R2 Score : 0.7022

Training Gradient Boosting...
MAE      : 2172.88
RMSE     : 2873.22
R2 Score : 0.6714


In [29]:
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

In [30]:
print("Training XGBoost...")

xgb_model.fit(
    X_train_transformed,
    y_train
)

xgb_pred = xgb_model.predict(X_test_transformed)

xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_mse = mean_squared_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(xgb_mse)
xgb_r2 = r2_score(y_test, xgb_pred)

print("\nXGBoost Results")
print("MAE :", round(xgb_mae, 2))
print("MSE :", round(xgb_mse, 2))
print("RMSE:", round(xgb_rmse, 2))
print("R2  :", round(xgb_r2, 4))

Training XGBoost...

XGBoost Results
MAE : 1940.88
MSE : 6820484.0
RMSE: 2611.61
R2  : 0.7286
